### **Import libraries**: 
- **Hugging Face** is a platform and library for utilizing machine learning models.
- **InferenceClient** is a library available on Hugging Face and is primarily used to perform inference – the process of reaching conclusions given a set of inputs.
- **Transformers** is a Python library created by Hugging Face that allows one to download, manipulate, and run thousands of pretrained, open-source AI models.
- **Pipeline** is a package within Transformers that simplifies the process of working with advanced AI models.
- **UUID** is a library that provides a way to generate universally unique identifiers. In this script, it is used to create identifiers for headlines.

In [49]:
import requests
from huggingface_hub import InferenceClient
from transformers import pipeline
import uuid
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date
from dateutil.relativedelta import relativedelta
from dateutil.rrule import rrule, DAILY
import random
import sys
import fastparquet
import os
from IPython.display import clear_output
import time

### **Create a data frame with ticker, headlineID, headline, article publication date, and article source columns.** ###

In [50]:
def package(ticker_lst_param):
    df_main = pd.DataFrame(columns = ["Ticker", "HeadlineID", "Headline", "Timestamp", "Source"])
    for element in ticker_lst_param:
        tickers = [element["Ticker"] for i in range(len(element["Headline"]))]
        headline_IDs = [str(uuid.uuid4())[:8] for i in range(len(element["Headline"]))]
        data = {"Ticker" : tickers, "HeadlineID" : headline_IDs, "Headline" : element["Headline"], 
                "Timestamp" : element["Timestamp"], "Source" : element["Source"]}
        df = pd.DataFrame(data = data)
        df_main = pd.concat([df_main, df])
    return df_main

### **Use the Alpha Vantage API to pull news associated with each ticker.**

### **To modify the tickers used, make changes to tickers.txt**

In [51]:
def get_data(txt_file_containing_tickers, collection_start_date, collection_end_date):
    with open(txt_file_containing_tickers, "r") as f:
        tickers = f.readline()
        tickers = tickers[1:-1]
        tickers = tickers.replace('"', "")
        tickers = tickers.split(",")
        tickers = [element.strip() for element in tickers]
    
    with open("api_keys/alpha_vantage.txt", "r") as f:
        alpha_vantage_api_key = f.readline().strip()

    limit = 50
    headline_data = []

    #tickers = ["AAPL", "NVDA"]
    
    for ticker in tickers:
        for day in rrule(DAILY, dtstart=collection_start_date, until=collection_end_date):
            time_from, time_to = day, day + timedelta(days = 1)
            time_from, time_to = time_from.strftime("%Y%m%dT%H%M"), time_to.strftime("%Y%m%dT%H%M")
            url = f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={ticker}&time_from={time_from}&time_to={time_to}&limit={limit}&apikey={alpha_vantage_api_key}"
            response = requests.get(url)
            #print(response.status_code)
            data = response.json()
            #print(ticker)
            #print(data)
            news = data["feed"]
            
            headlines = [article["title"] for article in news]

            #if ticker == "QQQ":
                #print(headlines)
            
            timestamps = [article["time_published"] for article in news]
            timestamps = [pd.to_datetime(timestamp, format = "mixed") for timestamp in timestamps]
            sources = [article["source"] for article in news]

            headlines_no_dupe, timestamps_no_dupe, sources_no_dupe = [], [], []


            for headline, timestamp, source in zip(headlines, timestamps, sources):
                if headline not in headlines_no_dupe:
                    headlines_no_dupe.append(headline)
                    timestamps_no_dupe.append(timestamp)
                    sources_no_dupe.append(source)


            #sys.exit()
            

            headline_datum = {"Ticker" : ticker, "Headline" : headlines_no_dupe, "Timestamp" : timestamps_no_dupe, "Source" : sources_no_dupe}
            headline_data.append(headline_datum)
        
        time.sleep(0.1)

    
        
    headline_data_packaged = package(headline_data)

    #print(headline_data_packaged[headline_data_packaged["Ticker"] == "QQQ"])


    #headline_data_packaged = headline_data_packaged[headline_data_packaged.groupby('Headline').cumcount().le(1)]

    

    #sys.exit()

    #print(headline_data_packaged[headline_data_packaged["Ticker"] == "BABA"])
    
    #headline_data_packaged.drop_duplicates(subset = "Headline", keep = "last", inplace = True)

    #headline_data_packaged = headline_data_packaged[headline_data_packaged.groupby('Headline').cumcount().le(1)]

    #print(headline_data_packaged[headline_data_packaged["Ticker"] == "QQQ"])

    #sys.exit()

    #print()
        
    #dataframe
    #print(headline_data_packaged[headline_data_packaged["Ticker"]=="NVDA"])
    reduce_articles(headline_data_packaged, tickers)

### **Reduce number of articles to maximum 25 per ticker per day.**

In [52]:
def reduce_articles(df, tickers):

    #print(df[df["Ticker"] == "BABA"])
    
    series_lst = []
    
    for ticker in tickers:
        if len(df[df["Ticker"]==ticker]) > 25:
            #print("Red", ticker)
            number_of_rows = len(df[df["Ticker"]==ticker])
        #print(type(df.index.values.tolist()))
            #print(df[df["Ticker"]==ticker])
            to_drop = random.sample(df[df["Ticker"]==ticker].index.values.tolist(), number_of_rows-25)
            series_lst.append(df[df["Ticker"]==ticker].drop(to_drop))
        else:
            series_lst.append(df[df["Ticker"]==ticker])

    df_concat = pd.concat(series_lst)

    df_concat = df_concat.reset_index(drop=True)

   # print(df_concat)

    df_concat.to_csv("test2.csv")

    #print(df_concat[df_concat["Ticker"] == "V"])









    
    # df.set_index(["Ticker", "HeadlineID"], inplace = True)
    
    # for ticker in tickers:
    #     number_of_rows = len(df.loc[ticker])
    #     if  number_of_rows > 25:
    #         to_drop = random.sample(df.loc[ticker].index.values.tolist(), number_of_rows - 25)
    #         sub_df = df.loc[ticker].drop(to_drop)
    #         #print(sub_df)
    #         #df.loc[ticker] = ser


    # print(sub_df)

    # df.drop(index = ("NVDA"), inplace = True)

    # df.loc["NVDA"] = np.nan

    # print(df)
    
    #print(len(df.loc[ticker]))
    #print(len(sub_df))

    
            
            
    #print(df.loc["NVDA"])
    #sys.exit()
    #print(df)
    # for ticker in tickers:
    #     if len(df[df["Ticker"]==ticker]) > 25:
    #         #print("Red", ticker)
    #         number_of_rows = len(df[df["Ticker"]==ticker])
    #     #print(type(df.index.values.tolist()))
    #         #print(df[df["Ticker"]==ticker])
    #         to_drop = random.sample(df[df["Ticker"]==ticker].index.values.tolist(), number_of_rows-25)
    #         df[df["Ticker"]==ticker] = df[df["Ticker"]==ticker].drop(to_drop)
    # print(df)

### **Utilize FinBERT - a pre-trained NLP model specialized for financial sentiment analysis – adapted by ProsusAI to analyze the probabilities (positive/negative/neutral) associated with each headline.**

In [53]:
def add_probabilities(ticker_news_data_df_param, api_key_param):
    pipe = pipeline("text-classification", model="ProsusAI/finbert")
    client = InferenceClient(provider="hf-inference", api_key=api_key_param)
    ticker_news_data_df_param[["pos", "neu", "neg"]] = 0.0, 0.0, 0.0
    ticker_news_data_df_param["Timestamp"] = pd.to_datetime(ticker_news_data_df_param["Timestamp"])
    ticker_news_data_df_param = ticker_news_data_df_param[ticker_news_data_df_param["Timestamp"] >= (datetime.today() - timedelta(days=22))]
    rows_num, row_counter = len(ticker_news_data_df_param), 1
    for index, row in ticker_news_data_df_param.iterrows():
        headline = row["Headline"]
        result = client.text_classification(headline, model="ProsusAI/finbert")
        for probability in result:
            row = row.copy()
            match probability["label"]:
                case "positive":
                    ticker_news_data_df_param.loc[index, "pos"] = probability.score
                case "neutral":
                    ticker_news_data_df_param.loc[index, "neu"] = probability.score
                case "negative":
                    ticker_news_data_df_param.loc[index, "neg"] = probability.score
        
        clear_output()
        print(str(round((row_counter/rows_num)*100, 2)) + "% complete.")
        row_counter +=1
        time.sleep(0.2)

    ticker_news_data_df_param.set_index(["Ticker", "Timestamp", "HeadlineID"], inplace = True)
    ticker_news_data_df_param.sort_index(inplace = True, level = 0)
    ticker_news_data_df_param.sort_index(inplace = True, level = 1, ascending = False, sort_remaining = False)
    return ticker_news_data_df_param

### **Main program used to collectively execute the whole script and retrieve the API key used for the inference package.**

In [54]:
def main():
    startTime = datetime.now()
    with open("api_keys/huggingface.txt", "r") as f:
        huggingface_api_key = f.readline().strip()



    #collection_start_date = pd.to_datetime("20260419", format="%Y%m%d") - relativedelta(years = 5)
    #collection_end_date = pd.to_datetime("20260419", format="%Y%m%d")

    collection_start_date = date(2026, 3, 9) #- relativedelta(years = 5)
    collection_end_date = date(2026, 3, 9)

    #print(collection_start_date, collection_end_date)
    
    

    tickers = get_data("tickers.txt", collection_start_date, collection_end_date)

    #print(collection_start_date, collection_end_date)

    #collection_end_date, years_to_collect = datetime.now(), 5
    #tickers = get_data("tickers.txt", collection_end_date, years_to_collect) ##YYYYMMDDTHHMM format
    
    #print(tickers)
    #ticker_news_data_df_with_probabilities = add_probabilities(ticker_news_data_df, api_key)
    #print(ticker_news_data_df_with_probabilities)
    #ticker_news_data_df_with_probabilities.to_parquet("daily_ticker_headlines_with_FinBERT_scoring_2.parquet")
    #print(datetime.now() - startTime)
main()